# InSAR Damage Assessment - PyGMTSAR Engine

An alternate processing engine for the pre1/pre2/post Damage Proxy Map pipeline in [insar-damage-assessment.ipynb](insar-damage-assessment.ipynb), using [PyGMTSAR](https://github.com/AlexeyPechnikov/pygmtsar) instead of ESA SNAP's `esa_snappy`/`gpt`.

Same method (Yun et al. 2015 Damage Proxy Map: coherence *drop* between a pre-event baseline pair and a co-event pair, not a raw coherence threshold), same Mosul AOI/scene triplet, different processing stack. Coherence-drop change detection is exactly PyGMTSAR's own [Kalkarindji Flooding example](https://github.com/AlexeyPechnikov/pygmtsar) ("Correlation loss serves to identify flooded areas") - this notebook follows that validated API sequence.

**Scope note:** this notebook reuses the Sentinel-1 SAFE products already downloaded by the SNAP notebook's CDSE pipeline (`data/input/s1/`) rather than re-implementing scene discovery/download - PyGMTSAR ships its own `ASF` downloader (Earthdata Login, not CDSE credentials) if starting from scratch, see the commented-out cell below. The point of this notebook is the processing engine, not the catalog step, which the SNAP notebook already handles.

Runs in a separate `pygmtsar` conda env (`pygmtsar` pip package - confirmed pure-Python, no GMTSAR C/csh binaries needed on Windows, unlike the older PyGMTSAR releases), kept isolated from the `eo` env so it can't disturb the working SNAP setup.

## 1. Setup & imports

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import dask
from dask.distributed import Client
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import folium
from folium.raster_layers import ImageOverlay
from scipy import ndimage

from pygmtsar import S1, Stack, Tiles, tqdm_dask
# from pygmtsar import ASF  # only needed if downloading scenes directly via Earthdata Login

## 2. Configuration & parameters

In [ ]:
# Shared with the SNAP notebook - same AOI, same already-downloaded Mosul scene triplet
input_dir = './data/input/s1/'
aoi_path = './data/aoi/mosul.geojson'

# PyGMTSAR working directory (aligned/intermediate products) and DEM cache - kept separate from
# the SNAP notebook's data/output/preprocessed so the two engines never collide on output files
workdir = './data/output/pygmtsar_work/mosul'
dem_path = './data/output/pygmtsar_work/mosul_dem.nc'
results_dir = './data/output/results_pygmtsar/'

# The pre1/pre2/post triplet already selected and downloaded by the SNAP notebook's
# select_images (baseline pair pre1/pre2, co-event pair pre2/post - the Damage Proxy Map method)
pre1_date = '20160923'
pre2_date = '20161005'
post_date = '20170620'

# Subswath IW2 as an integer, GMTSAR convention (1/2/3, not the SNAP notebook's 'IW1'/'IW2'/'IW3'
# strings) - same subswath independently confirmed by the SNAP notebook's per-burst footprint
# auto-selection (get_subswath_footprint/select_subswath in that notebook's step 11)
subswath = 2

# Damage Proxy Map threshold: flag pixels where co-event coherence drops by more than this,
# relative to the pre-event baseline pair - same value and same reasoning as the SNAP notebook
coherence_drop_threshold = 0.25
# Minimum connected pixel count to keep a damage candidate cluster (drops speckle-noise pixels),
# mirroring the SNAP notebook's connectedPixelCount-style filter
min_cluster_pixels = 8

aoi = gpd.read_file(aoi_path)
aoi_geom = aoi.geometry.iloc[0]
aoi_centroid = aoi_geom.centroid

os.makedirs(os.path.dirname(dem_path), exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

## 3. (Optional) Download scenes via PyGMTSAR's own ASF downloader

Not run here - the triplet above is already present in `data/input/s1/` from the SNAP notebook's CDSE pipeline. Left for reference if starting this notebook from scratch without that data. Needs an Earthdata Login (different credentials than CDSE), not CDSE_USERNAME/CDSE_PASSWORD.

In [ ]:
# from pygmtsar import ASF
# asf = ASF(asf_username, asf_password)  # or ASF(None, None) to be prompted interactively
# scene_ids = [
#     f'S1A_IW_SLC__1SDV_{pre1_date}T030941_...',  # full scene IDs from the CDSE catalog query
#     f'S1A_IW_SLC__1SDV_{pre2_date}T030941_...',
#     f'S1B_IW_SLC__1SDV_{post_date}T030853_...',
# ]
# asf.download(input_dir, scene_ids, subswath)

## 4. Verify the local scene triplet

In [ ]:
# Scan the shared input directory for SLC scenes at the target subswath, then keep only the
# three dates this run cares about (the directory may accumulate scenes from other runs/AOIs).
# polarization='VV' matches the SNAP graph's selectedPolarisations>VV< - scan_slc otherwise
# returns one row per polarization (VH+VV) rather than one row per scene.
all_scenes = S1.scan_slc(input_dir, subswath=subswath, polarization='VV')
target_dates = {pre1_date, pre2_date, post_date}
scenes = all_scenes[all_scenes.index.map(lambda d: pd.Timestamp(d).strftime('%Y%m%d') in target_dates)]
scenes = scenes.sort_index()

assert len(scenes) == 3, (
    f'Expected exactly 3 scenes (pre1/pre2/post), found {len(scenes)}. '
    'Check input_dir and the pre1_date/pre2_date/post_date values above.'
)
scenes

## 5. DEM

In [ ]:
# PyGMTSAR downloads its own DEM (NetCDF, not the SNAP notebook's GeoTIFF cache) via a separate
# Python-native path - unaffected by the SNAP/JVM proxy issue diagnosed in the other notebook
if not os.path.exists(dem_path):
    Tiles().download_dem(aoi_geom, filename=dem_path)
xr.open_dataarray(dem_path).plot.imshow(cmap='terrain')

## 6. Orbits

In [ ]:
# Downloads any missing precise orbit (.EOF) files for the scanned scenes
S1.download_orbits(input_dir, scenes)

## 7. Local Dask cluster

In [ ]:
if 'client' in globals():
    client.close()
client = Client()
client

## 8. Initialize the PyGMTSAR Stack

In [ ]:
sbas = Stack(workdir, drop_if_exists=True).set_scenes(scenes)
sbas.to_dataframe()

## 8b. Reframe scenes to the AOI

Crops each scene's radar-coordinate extent down to the AOI before alignment/interferometry - 
the PyGMTSAR equivalent of the SNAP graph's wktAoi-based TOPSAR-Split cropping. Matters for 
speed: the SNAP run over the full, uncropped subswath width took multiple minutes per pair; 
skipping this step here would process the same full-width data.

In [ ]:
sbas.compute_reframe(aoi_geom)

In [ ]:
sbas.load_dem(dem_path, aoi_geom)

## 9. Align (coregister) the scene triplet

In [ ]:
sbas.compute_align()

## 10. Geocoding transform

Builds the radar-to-map coordinate lookup table PyGMTSAR uses later to reproject coherence results with `ra2ll`. The resolution argument follows the validated PyGMTSAR flooding example as-is (an approximate grid resolution in meters, not independently re-derived here).

In [ ]:
sbas.compute_geocode(45.)

## 11. Build the baseline/co-event pairs

With exactly 3 chronologically-sorted scenes (pre1 < pre2 < post), consecutive pairing gives exactly the two pairs the Damage Proxy Map method needs: pre1-pre2 (baseline, no conflict in between) and pre2-post (co-event). Pair index 0 is always the baseline pair, index 1 the co-event pair - this only holds because the triplet is exactly 3 scenes, asserted in step 4.

In [ ]:
df = sbas.to_dataframe()
pairs = np.asarray([df.index[:-1], df.index[1:]])
pairs  # pairs[:, 0] = baseline (pre1/pre2), pairs[:, 1] = co-event (pre2/post)

## 12. Compute interferograms and coherence for both pairs

In [ ]:
topo = sbas.get_topo()
data = sbas.open_data()
intensity = sbas.multilooking(np.square(np.abs(data)), wavelength=400, coarsen=(3, 12))
phase = sbas.phasediff(pairs, data, topo)
phase = sbas.multilooking(phase, wavelength=400, coarsen=(3, 12))
corr = sbas.correlation(phase, intensity)
tqdm_dask(corr := dask.persist(corr)[0], desc='Compute Correlation')
corr

In [ ]:
# Geocode radar-coordinate coherence to lat/lon
corr_ll = sbas.ra2ll(corr)
corr_ll

## 13. Detect damage candidates (coherence drop)

In [ ]:
baseline_coh = corr_ll[0]  # pre1/pre2 - normal, non-conflict decorrelation
coevent_coh = corr_ll[1]   # pre2/post - includes conflict-driven decorrelation
coherence_drop = baseline_coh - coevent_coh

damage_candidate = (coherence_drop > coherence_drop_threshold).values

# Drop speckle-noise pixels: keep only clusters with at least min_cluster_pixels connected pixels,
# mirroring the SNAP notebook's connectedPixelCount-style filter
labeled, n_features = ndimage.label(damage_candidate)
sizes = ndimage.sum(damage_candidate, labeled, range(1, n_features + 1))
keep = np.zeros(n_features + 1, dtype=bool)
keep[1:] = sizes >= min_cluster_pixels
damage_mask = keep[labeled]

affected_fraction = damage_mask.mean()
print(f'{affected_fraction:.2%} of the AOI flagged as damage-candidate '
      f'(coherence drop > {coherence_drop_threshold}, >= {min_cluster_pixels} connected pixels)')

## 14. Static comparison figure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
baseline_coh.plot.imshow(ax=axes[0], cmap='gray', vmin=0, vmax=1, add_colorbar=False)
axes[0].set_title('Baseline coherence (pre1/pre2)')
coevent_coh.plot.imshow(ax=axes[1], cmap='gray', vmin=0, vmax=1, add_colorbar=False)
axes[1].set_title('Co-event coherence (pre2/post)')
axes[2].imshow(damage_mask, cmap='Reds')
axes[2].set_title(f'Damage candidates ({affected_fraction:.1%} of AOI)')
plt.tight_layout()
plt.show()

## 15. Interactive damage map

In [ ]:
lat_min, lat_max = float(corr_ll.lat.min()), float(corr_ll.lat.max())
lon_min, lon_max = float(corr_ll.lon.min()), float(corr_ll.lon.max())
bounds = [[lat_min, lon_min], [lat_max, lon_max]]

fmap = folium.Map(location=[aoi_centroid.y, aoi_centroid.x], zoom_start=12, tiles='CartoDB dark_matter')

coh_cmap = plt.colormaps['gray']
coevent_rgba = coh_cmap(np.clip(coevent_coh.values, 0, 1))
ImageOverlay(coevent_rgba, bounds=bounds, opacity=0.8, name='Co-event coherence').add_to(fmap)

damage_rgba = np.zeros((*damage_mask.shape, 4))
damage_rgba[damage_mask] = [1, 0, 0, 0.6]
ImageOverlay(damage_rgba, bounds=bounds, opacity=1.0, name='Damage candidates').add_to(fmap)

folium.LayerControl().add_to(fmap)
fmap

## 16. Save georeferenced outputs

In [ ]:
run_dir = os.path.join(results_dir, f'{pre1_date}_{pre2_date}_{post_date}')
os.makedirs(run_dir, exist_ok=True)

# Static comparison figure
fig.savefig(os.path.join(run_dir, 'coherence_damage_comparison.png'), dpi=150, bbox_inches='tight')

# Interactive map
fmap.save(os.path.join(run_dir, 'damage_map.html'))

# 3-band GeoTIFF: baseline coherence, co-event coherence, damage mask - via rioxarray, reusing
# corr_ll's geocoding (written by ra2ll) rather than re-deriving a transform by hand
out_ds = xr.concat(
    [baseline_coh, coevent_coh, xr.DataArray(damage_mask.astype('float32'), coords=baseline_coh.coords)],
    dim=pd.Index(['baseline_coherence', 'coevent_coherence', 'damage_mask'], name='band'),
)
out_ds = out_ds.rio.write_crs('EPSG:4326')
out_ds.rio.to_raster(os.path.join(run_dir, 'damage_assessment.tif'))

# Summary stats
summary = {
    'pre1_date': pre1_date, 'pre2_date': pre2_date, 'post_date': post_date,
    'subswath': subswath, 'coherence_drop_threshold': coherence_drop_threshold,
    'min_cluster_pixels': min_cluster_pixels, 'affected_fraction': float(affected_fraction),
}
with open(os.path.join(run_dir, 'summary_stats.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Results written to {run_dir}')